# 06 · Label extraction audit

Summarises extraction paths (JSON, embedded JSON, fallback, unknown) and how often rationales mention other labels, then draws a stratified sample of 100 outputs per condition into a CSV for **manual checking**. Fill in `manual_label` where the parsed label is wrong and save as `*_checked.csv`; notebook 07 applies the corrections.

In [1]:
# ==== Setup (run first) ====
# SOURCE = "drive"  : the repository folder is on Google Drive (use this now)
# SOURCE = "github" : clone from GitHub (after the repository is public)
SOURCE = "drive"
DRIVE_REPO = "/content/drive/MyDrive/2026/Publikasi Sinta 2 Baru/mad-indo-sentiment"   # folder from the unzipped package
REPO_URL = "https://github.com/deepittalk-repo/mad-indo-sentiment.git"   # used only when SOURCE = "github"

import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")          # results persist in MyDrive/mad-indo-results
    if SOURCE == "drive":
        REPO_DIR = DRIVE_REPO
        assert os.path.exists(REPO_DIR), f"Folder not found: {REPO_DIR}. Upload the unzipped folder to MyDrive first."
    else:
        REPO_DIR = "/content/mad-indo-sentiment"
        if not os.path.exists(REPO_DIR):
            !git clone -q $REPO_URL $REPO_DIR
    %cd $REPO_DIR
    !pip -q install -r requirements.txt
else:
    # Local run: start Jupyter from the repository root or the notebooks/ folder.
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
sys.path.insert(0, "src")

from mad_indo import data
cfg = data.load_config()
RESULTS = data.results_dir()
print("Results folder:", RESULTS)
LIMIT = cfg["limit_test_samples"]   # null in config = full test set


Mounted at /content/drive
/content/drive/MyDrive/2026/Publikasi Sinta 2 Baru/mad-indo-sentiment
Results folder: /content/drive/MyDrive/mad-indo-results


In [2]:
import pandas as pd
from mad_indo.data import read_jsonl
rows = []
for f in sorted((RESULTS / "single_agent").glob("*.jsonl")):
    for r in read_jsonl(f):
        rows.append({"condition": f"{r['model']}|{r['method']}|p{r['paraphrase']}|s{r['seed']}", "task": r["task"],
                     "id": r["id"], "gold": r["gold"], "pred": r["pred"], "path": r["path"],
                     "mentions_other": bool(r["mentions_other"]), "rationale": r["rationale"], "raw": r["raw"], "agent": ""})
for f in sorted((RESULTS / "debate").glob("*.jsonl")):
    for r in read_jsonl(f):
        for k, rd in enumerate(r["rounds"]):
            for i, a in enumerate(rd):
                rows.append({"condition": f"{r['composition']}|round{k}|s{r['seed']}", "task": r["task"], "agent": str(i),
                             "id": r["id"], "gold": r["gold"], "pred": a["label"], "path": a["path"],
                             "mentions_other": bool(a["mentions_other"]), "rationale": a["rationale"], "raw": ""})
A = pd.DataFrame(rows)
audit = A.groupby(["task", "condition"]).agg(n=("id", "size"),
        json=("path", lambda s: (s == "json").mean()), fallback=("path", lambda s: (s == "fallback_first_mention").mean()),
        unknown=("path", lambda s: (s == "unknown").mean()), mentions_other=("mentions_other", "mean")).round(3)
audit.to_csv(RESULTS / "audit_summary.csv"); audit

n   json  fallback  unknown  mentions_other
task condition                                                               
emot gemma|S1_zero_shot|p0|s11  440  1.000     0.000      0.0           0.109
     gemma|S2_few_shot|p0|s11   440  1.000     0.000      0.0           0.039
     gemma|S2_few_shot|p0|s22   440  1.000     0.000      0.0           0.045
     gemma|S2_few_shot|p0|s33   440  1.000     0.000      0.0           0.052
     gemma|S3_cot|p0|s11        440  1.000     0.000      0.0           0.109
...                             ...    ...       ...      ...             ...
smsa qwen|S2_few_shot|p2|s22    500  0.998     0.002      0.0           0.064
     qwen|S2_few_shot|p2|s33    500  1.000     0.000      0.0           0.060
     qwen|S3_cot|p0|s11         500  1.000     0.000      0.0           0.054
     qwen|S3_cot|p1|s11         500  1.000     0.000      0.0           0.040
     qwen|S3_cot|p2|s11         500  1.000     0.000      0.0           0.056

[97 rows x 5 columns]

In [3]:
# Stratified manual-check sample: oversample risky outputs (fallback or rationale naming other labels)
n = cfg["audit"]["sample_per_condition"]
samples = []
for (task, cond), g in A.groupby(["task", "condition"]):
    risky, rest = g[(g.path != "json") | g.mentions_other], g[(g.path == "json") & ~g.mentions_other]
    k = min(len(risky), n // 2)
    s = pd.concat([risky.sample(k, random_state=0), rest.sample(min(len(rest), n - k), random_state=0)])
    samples.append(s.assign(manual_label=""))
check = pd.concat(samples)
check.to_csv(RESULTS / "audit_manual_check.csv", index=False)
print(len(check), "rows written to", RESULTS / "audit_manual_check.csv")

8760 rows written to /content/drive/MyDrive/mad-indo-results/audit_manual_check.csv


After the manual check, save the file as `audit_manual_check_checked.csv` in the same folder and run the cell below. Original labels are kept (`pred_original` / `label_original`); D0 and vote are recomputed for corrected debates.

In [4]:
from mad_indo.audit import apply_corrections
checked = RESULTS / "audit_manual_check_checked.csv"
if checked.exists():
    log = apply_corrections(RESULTS, checked); log.to_csv(RESULTS / "audit_corrections_log.csv", index=False)
    print(len(log), "labels corrected"); display(log.head())
else:
    print("No checked file yet:", checked)

No checked file yet: /content/drive/MyDrive/mad-indo-results/audit_manual_check_checked.csv
